# Trích xuất embedding (extract_embedding): Qwen3-VL-Embedding-2B qua vLLM

Chạy các cell theo thứ tự. Notebook liệt kê ZIP trong Drive nguồn, chọn `DATASET` (V3C hoặc LHE) và khoảng ID bằng `START_VIDEO_ID` và `END_VIDEO_ID` (bao gồm hai đầu), rồi xử lý lần lượt trên **một server vLLM**. ZIP nguồn `00685.zip` có các ảnh `000000.jpg`, `000001.jpg` ở ngay gốc. ZIP kết quả được upload vào `keyframe/V3C/00685.zip`, bên trong chứa `00685/000000.npy`, `00685/000001.npy`; LHE dùng `keyframe/LHE/LHE01.zip` và `LHE01/000000.npy`. Dùng cùng root embedding với notebook ASR/caption. Mỗi video chỉ được upload sau khi kiểm tra đủ vector 2048 chiều.

Bật Internet và GPU trong Kaggle. vLLM 0.32.0 nằm trong môi trường Python 3.12 riêng để không thay PyTorch/CUDA của kernel. Điền hai Drive folder ID hoặc tạo Kaggle Secrets cùng tên. `rclone config file` cho biết config mặc định; notebook sao chép `RCLONE_CONFIG_SOURCE` từ Kaggle Input sang file ghi được ở `/kaggle/working` trước khi dùng.


In [ ]:
# 1. Cấu hình
import os
import re
import shutil
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/iamtro-2006/event-retrieval-system.git"
REPO_BRANCH = "vbs-dev"
MODEL_KEY = "qwen3_vl_embedding_2b"
MODEL_ID = "Qwen/Qwen3-VL-Embedding-2B"
DATASET = "V3C"              # đổi thành "LHE" để xử lý LHE01.zip, LHE02.zip...
START_VIDEO_ID = 11             # V3C: 00011.zip; LHE: LHE11.zip
END_VIDEO_ID = 20               # tính cả hai đầu
ARCHIVE_REGEX = r"^LHE\d{2}\.zip$" if DATASET == "LHE" else r"^\d{5}\.zip$"
def video_name(number):
    return f"LHE{number:02d}" if DATASET == "LHE" else f"{number:05d}"
FRAME_REGEX = r"^\d{6}$"     # 000000.jpg -> 000000.npy
BATCH_SIZE = 4                  # số request ảnh đồng thời tới vLLM
MAX_MODEL_LEN = 8192
VLLM_PORT = 8001
VLLM_URL = f"http://127.0.0.1:{VLLM_PORT}/v1"
RCLONE_CONFIG_SOURCE = Path("/kaggle/input/datasets/nguyennhattan121106/rclone-pro-pro/rclone.conf")
RCLONE_REMOTE = "iamtro"
DOWNLOAD_ROOT_FOLDER_ID = ""    # hoặc Kaggle Secret cùng tên
UPLOAD_ROOT_FOLDER_ID = ""      # Root chung embeddings_qwen3_vl_embedding_2b_vllm cho keyframe/asr/caption

def secret(name, value):
    if value:
        return value
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise ValueError(f"Thiếu {name}; điền ở cell này hoặc Kaggle Secrets") from exc

DOWNLOAD_ROOT_FOLDER_ID = secret("DOWNLOAD_ROOT_FOLDER_ID", DOWNLOAD_ROOT_FOLDER_ID)
UPLOAD_ROOT_FOLDER_ID = secret("UPLOAD_ROOT_FOLDER_ID", UPLOAD_ROOT_FOLDER_ID)
if DOWNLOAD_ROOT_FOLDER_ID == UPLOAD_ROOT_FOLDER_ID:
    raise ValueError("Drive folder nguồn và đích phải khác nhau để không ghi đè ZIP keyframe")
for name, pattern in (("ARCHIVE_REGEX", ARCHIVE_REGEX), ("FRAME_REGEX", FRAME_REGEX)):
    try:
        re.compile(pattern)
    except re.error as exc:
        raise ValueError(f"{name} không hợp lệ: {exc}") from exc
if DATASET not in {"V3C", "LHE"}:
    raise ValueError("DATASET phải là V3C hoặc LHE")
max_id = 99 if DATASET == "LHE" else 99999
if (not isinstance(START_VIDEO_ID, int) or not isinstance(END_VIDEO_ID, int)
        or START_VIDEO_ID < 1 or END_VIDEO_ID < START_VIDEO_ID or END_VIDEO_ID > max_id):
    raise ValueError(f"Cần 1 <= START_VIDEO_ID <= END_VIDEO_ID <= {max_id}")
if not isinstance(BATCH_SIZE, int) or BATCH_SIZE < 1 or not isinstance(MAX_MODEL_LEN, int) or MAX_MODEL_LEN < 1:
    raise ValueError("BATCH_SIZE/MAX_MODEL_LEN phải là số nguyên dương")
if not RCLONE_CONFIG_SOURCE.is_file():
    raise FileNotFoundError(f"Không thấy rclone config: {RCLONE_CONFIG_SOURCE}")

RUN_ROOT = Path(tempfile.mkdtemp(prefix=f"embedding-{DATASET.lower()}-", dir="/kaggle/working"))
RCLONE_CONFIG = RUN_ROOT / "rclone.conf"
shutil.copyfile(RCLONE_CONFIG_SOURCE, RCLONE_CONFIG)
RCLONE_CONFIG.chmod(0o600)  # rclone cần ghi token mới; /kaggle/input là read-only
REPO_DIR = RUN_ROOT / "event-retrieval-system"
BACKEND_DIR = REPO_DIR / "backend"
VIDEO_WORK_ROOT = RUN_ROOT / "videos"
VIDEO_WORK_ROOT.mkdir()
print("Run directory:", RUN_ROOT)


In [ ]:
# 2. Clone nhánh, chuẩn bị vLLM và rclone (có thể chạy lại cell này sau lỗi tải)
import importlib.util
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile

def run(args, *, cwd=None, env=None):
    args = [str(item) for item in args]
    print("+", " ".join(args[:4]), "..." if len(args) > 4 else "")
    return subprocess.run(args, cwd=cwd, env=env, check=True, text=True)

if not REPO_DIR.exists():
    run(["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
branch = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if branch != REPO_BRANCH or not (BACKEND_DIR / "configs/embeddings.yaml").is_file():
    raise RuntimeError(f"Repo/branch không đúng: {branch}")
requirements_vllm = BACKEND_DIR / "requirements-vllm.txt"
if not requirements_vllm.is_file() or not any(line.strip() == "vllm==0.32.0" for line in requirements_vllm.read_text().splitlines()):
    raise RuntimeError("Repo không pin vLLM 0.32.0 như notebook yêu cầu")
print("Checked out:", branch, commit)

# Kernel giữ nguyên torch/CUDA của Kaggle. vLLM có venv riêng; hai bên giao tiếp HTTP.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Cần bật GPU trong Kaggle Settings > Accelerator")
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__)
run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade-strategy", "only-if-needed",
     "PyYAML>=6,<7", "httpx>=0.27,<1", "pillow>=10,<13", "tqdm>=4.66,<5", "uv"])
if importlib.util.find_spec("cv2") is None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "opencv-python-headless>=4.9,<5"])
import yaml
import httpx
import cv2
from PIL import Image
import tqdm
if not torch.cuda.is_available():
    raise RuntimeError("Torch mất CUDA sau khi cài dependency")

VLLM_VENV = RUN_ROOT / "venv-vllm"
VLLM_PYTHON = VLLM_VENV / "bin/python"
if not VLLM_PYTHON.is_file():
    run([sys.executable, "-m", "uv", "python", "install", "3.12"])
    run([sys.executable, "-m", "uv", "venv", "--python", "3.12", VLLM_VENV])
if not (VLLM_VENV / "bin/vllm").is_file():
    run([sys.executable, "-m", "uv", "pip", "install", "--python", VLLM_PYTHON,
         "--torch-backend=auto", "-r", requirements_vllm])
run([VLLM_PYTHON, "-c", "import vllm; print('vLLM:', vllm.__version__)"],
    env={**os.environ, "CUDA_VISIBLE_DEVICES": ""})

# downloads.rclone.org có thể trả 403 trên Kaggle. Ưu tiên binary sẵn có,
# sau đó apt của hệ thống; nếu apt không dùng được thì thử release GitHub.
rclone = shutil.which("rclone")
if rclone is None and shutil.which("apt-get") and os.geteuid() == 0:
    apt = shutil.which("apt-get")
    apt_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    try:
        install = subprocess.run([apt, "install", "-y", "-qq", "rclone"], env=apt_env,
                                 capture_output=True, text=True, timeout=300)
        if install.returncode != 0:
            subprocess.run([apt, "update", "-qq"], env=apt_env, check=True,
                           capture_output=True, text=True, timeout=300)
            run([apt, "install", "-y", "-qq", "rclone"], env=apt_env)
        rclone = shutil.which("rclone")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        print("apt không cài được rclone:", exc)

if rclone is None:
    release = "v1.75.2"
    archive = f"rclone-{release}-linux-amd64.zip"
    url = f"https://github.com/rclone/rclone/releases/download/{release}/{archive}"
    rclone_zip = RUN_ROOT / archive
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "kaggle-rclone-setup"})
        with urllib.request.urlopen(request, timeout=120) as src, rclone_zip.open("wb") as dst:
            shutil.copyfileobj(src, dst)
        with zipfile.ZipFile(rclone_zip) as zf:
            matches = [item for item in zf.infolist() if item.filename.endswith("/rclone") and not item.is_dir()]
            if len(matches) != 1:
                raise RuntimeError("Release rclone không chứa đúng một binary")
            binary = RUN_ROOT / "rclone"
            with zf.open(matches[0]) as src, binary.open("wb") as dst:
                shutil.copyfileobj(src, dst)
        binary.chmod(0o755)
        rclone = str(binary)
    except Exception as exc:
        raise RuntimeError("Không cài được rclone qua apt hoặc GitHub release. "
                           "Hãy gắn rclone binary vào Kaggle và thêm nó vào PATH.") from exc
run([rclone, "version"])


In [ ]:
# 3. Liệt kê ZIP nguồn và chọn hàng loạt theo pattern
import re
import subprocess

listing = subprocess.run(
    [str(rclone), "--config", str(RCLONE_CONFIG), "--drive-root-folder-id", DOWNLOAD_ROOT_FOLDER_ID,
     "lsf", f"{RCLONE_REMOTE}:", "--files-only", "--format", "p"],
    check=True, capture_output=True, text=True,
)
names = [line.strip() for line in listing.stdout.splitlines() if line.strip()]
selected_ids = [video_name(number) for number in range(START_VIDEO_ID, END_VIDEO_ID + 1)]
selected_names = {f"{video_id}.zip" for video_id in selected_ids}
matching = [name for name in names if name in selected_names and re.fullmatch(ARCHIVE_REGEX, name)]
if len(matching) != len(set(matching)):
    raise RuntimeError("Drive trả về ZIP trùng tên; dừng để tránh ghi đè")
archives = [f"{video_id}.zip" for video_id in selected_ids if f"{video_id}.zip" in matching]
if not archives:
    raise FileNotFoundError(f"Không có ZIP trong khoảng ID {START_VIDEO_ID}–{END_VIDEO_ID} ở Drive nguồn")
archive_set = set(archives)
missing = [f"{video_id}.zip" for video_id in selected_ids
           if f"{video_id}.zip" not in archive_set]
if missing:
    print(f"Không thấy {len(missing)} ZIP trong khoảng: {missing[:10]}")
print(f"Sẽ xử lý {len(archives)} ZIP, ID {START_VIDEO_ID}–{END_VIDEO_ID}: {archives[:10]}")


In [ ]:
# 4. Khởi động một server vLLM cho toàn bộ batch
import copy
import os
import subprocess
import time
import yaml
import httpx

cfg_template = yaml.safe_load((BACKEND_DIR / "configs/embeddings.yaml").read_text(encoding="utf-8"))
matches = [m for m in cfg_template.get("models", []) if m.get("key") == MODEL_KEY]
if len(matches) != 1 or matches[0].get("backend") != "qwen3_vl_embedding" or matches[0].get("name") != MODEL_ID:
    raise ValueError(f"Repo không có đúng model Qwen3 {MODEL_KEY}")
model = copy.deepcopy(matches[0])
model.update({"enabled": True, "engine": "vllm", "vllm_url": VLLM_URL,
              "batch_size": BATCH_SIZE, "precision": "fp16", "device": "auto", "normalize": False})

server_log_path = RUN_ROOT / "vllm.log"
server_log = server_log_path.open("w", encoding="utf-8")
server = subprocess.Popen(
    [str(VLLM_VENV / "bin/vllm"), "serve", MODEL_ID, "--runner", "pooling",
     "--max-model-len", str(MAX_MODEL_LEN), "--host", "127.0.0.1", "--port", str(VLLM_PORT)],
    stdout=server_log, stderr=subprocess.STDOUT, env=os.environ.copy(),
)
server_log.close()

deadline = time.monotonic() + 1800
while time.monotonic() < deadline:
    if server.poll() is not None:
        raise RuntimeError(f"vLLM thoát sớm (code {server.returncode}). Xem {server_log_path}\n"
                           + "\n".join(server_log_path.read_text(errors="replace").splitlines()[-30:]))
    try:
        response = httpx.get(f"{VLLM_URL}/models", timeout=5)
        response.raise_for_status()
        served = {item.get("id") for item in response.json().get("data", [])}
        if MODEL_ID in served:
            break
        raise RuntimeError(f"Server tại {VLLM_URL} đang phục vụ model khác: {served}")
    except (httpx.HTTPError, ValueError):
        time.sleep(5)
else:
    server.terminate()
    raise TimeoutError(f"vLLM chưa sẵn sàng sau 30 phút. Xem {server_log_path}\n"
                       + "\n".join(server_log_path.read_text(errors="replace").splitlines()[-30:]))
print("vLLM ready:", MODEL_ID, VLLM_URL)


In [ ]:
# 5. Mỗi ZIP nguồn -> kiểm tra vector -> ZIP <id>/<frame>.npy -> upload
import copy
import json
import shutil
import stat
import subprocess
import sys
import zipfile
from pathlib import Path
import numpy as np
import yaml

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp"}
engine_env = os.environ.copy()
engine_env["PYTHONPATH"] = str(BACKEND_DIR)
engine_env["QWEN_EMBEDDING_ENGINE"] = "vllm"
engine_env["QWEN_VLLM_URL"] = VLLM_URL
engine_env["TOKENIZERS_PARALLELISM"] = "false"

def process_archive(archive_name):
    video_id = archive_name[:-4]
    work = VIDEO_WORK_ROOT / video_id
    if work.exists():
        if work.resolve().parent != VIDEO_WORK_ROOT.resolve():
            raise RuntimeError("Đường dẫn dọn dẹp nằm ngoài VIDEO_WORK_ROOT")
        shutil.rmtree(work)  # cho phép chạy lại video đã lỗi trong cùng phiên
    work.mkdir()
    source_zip = work / "source" / archive_name
    source_zip.parent.mkdir()
    run([rclone, "--config", RCLONE_CONFIG, "--drive-root-folder-id", DOWNLOAD_ROOT_FOLDER_ID,
         "copyto", f"{RCLONE_REMOTE}:{archive_name}", source_zip])
    if not source_zip.is_file() or source_zip.stat().st_size == 0:
        raise RuntimeError(f"ZIP tải về rỗng: {archive_name}")

    image_dir = work / "keyframes" / DATASET / video_id
    image_dir.mkdir(parents=True)
    with zipfile.ZipFile(source_zip) as zf:
        bad = zf.testzip()
        if bad:
            raise ValueError(f"ZIP {archive_name} bị lỗi CRC ở {bad}")
        images = []
        for info in zf.infolist():
            if info.is_dir():
                continue
            path = Path(info.filename)
            if path.suffix.lower() not in IMAGE_EXT:
                continue
            if len(path.parts) != 1 or "\\" in info.filename or stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f"Ảnh phải nằm ngay gốc ZIP, không có symlink: {info.filename}")
            if not re.fullmatch(FRAME_REGEX, path.stem):
                raise ValueError(f"Tên frame không khớp FRAME_REGEX: {info.filename}")
            images.append(info)
        stems = [Path(info.filename).stem for info in images]
        if not images or len(stems) != len(set(stems)):
            raise ValueError(f"ZIP {archive_name} không có ảnh hoặc có frame trùng tên")
        for info in images:
            with zf.open(info) as src, (image_dir / info.filename).open("wb") as dst:
                shutil.copyfileobj(src, dst)

    cfg = copy.deepcopy(cfg_template)
    cfg["models"] = [copy.deepcopy(model)]
    cfg["input_keyframes_root"] = str(work / "keyframes")
    cfg["output_embeddings_root"] = str(work / "output")
    cfg.setdefault("save", {}).update({"overwrite": False, "extension": ".npy"})
    config_path = work / "embeddings.runtime.yaml"
    config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    run([sys.executable, "scripts/embedding_extraction/run.py", "--config", config_path,
         "--model", MODEL_KEY], cwd=BACKEND_DIR, env=engine_env)

    vector_dir = work / "output" / f"embeddings_{MODEL_KEY}_vllm" / "keyframe" / DATASET / video_id
    expected = {f"{stem}.npy" for stem in stems}
    actual = {p.name for p in vector_dir.glob("*.npy")}
    if actual != expected:
        raise RuntimeError(f"{video_id}: vector thiếu={sorted(expected-actual)[:10]}, thừa={sorted(actual-expected)[:10]}")
    vectors = sorted(vector_dir.glob("*.npy"))
    for path in vectors:
        vector = np.load(path, allow_pickle=False)
        if vector.shape != (2048,) or vector.dtype != np.float32 or not np.isfinite(vector).all():
            raise ValueError(f"Vector không hợp lệ: {path}, shape={vector.shape}, dtype={vector.dtype}")
        if not np.isclose(np.linalg.norm(vector), 1.0, atol=1e-3, rtol=0):
            raise ValueError(f"Vector chưa chuẩn hóa: {path}")

    output_zip = work / "result" / f"{video_id}.zip"
    output_zip.parent.mkdir()
    with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
        for path in vectors:
            zf.write(path, arcname=f"{video_id}/{path.name}")
    with zipfile.ZipFile(output_zip) as zf:
        expected_names = {f"{video_id}/{name}" for name in expected}
        if zf.testzip() or set(zf.namelist()) != expected_names:
            raise RuntimeError(f"ZIP embedding {video_id} không hợp lệ")

    run([rclone, "--config", RCLONE_CONFIG, "--drive-root-folder-id", UPLOAD_ROOT_FOLDER_ID,
         "copyto", output_zip, f"{RCLONE_REMOTE}:keyframe/{DATASET}/{video_id}.zip"])
    if work.resolve().parent != VIDEO_WORK_ROOT.resolve():
        raise RuntimeError("Đường dẫn dọn dẹp nằm ngoài VIDEO_WORK_ROOT")
    shutil.rmtree(work)  # chỉ sau khi upload thành công, tránh đầy ổ Kaggle
    return len(images)

completed = []
failed = {}
try:
    for index, archive_name in enumerate(archives, 1):
        print(f"[{index}/{len(archives)}] {archive_name}")
        try:
            count = process_archive(archive_name)
            completed.append((archive_name, count))
            print(f"Uploaded {archive_name}: {count} vectors")
        except subprocess.CalledProcessError as exc:
            failed[archive_name] = f"Lệnh ngoài thất bại (exit {exc.returncode})"
            print(f"FAILED {archive_name}: lệnh ngoài thất bại; dừng batch để tránh lặp lỗi")
            break
        except Exception as exc:
            failed[archive_name] = str(exc)
            print(f"FAILED {archive_name}: {exc}")
finally:
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()

summary = {"completed": completed, "failed": failed, "model": MODEL_ID,
           "engine": "vllm", "dataset": DATASET, "version": "0.32.0", "commit": commit}
(RUN_ROOT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Hoàn tất {len(completed)}/{len(archives)} ZIP. Summary: {RUN_ROOT / 'summary.json'}")
if failed:
    raise RuntimeError(f"Có {len(failed)} ZIP lỗi: {list(failed)}; xem summary và thư mục run để xử lý lại")
